# Reinforcement Learning from Scratch: Teaching an Agent to Escape a Maze

In this notebook you'll build a **Q-learning** agent with nothing but Python and NumPy.
It starts knowing nothing, tries things, gets rewarded or punished, and slowly figures out
the best path to the goal.

**How to run it:** click on a code cell and press **Shift + Enter** to run it and move to the next one.
Run the cells in order, top to bottom. If anything breaks, use **Restart** then **Run All** at the top.

| Part | What happens |
|---|---|
| 1 | Meet the world (states, actions, rewards) |
| 2 | Watch a totally random agent |
| 3 | Build the agent's brain: the Q-table |
| 4 | Train it |
| 5 | See what it learned |
| 6 | Your turn: experiments |

In [ ]:
import numpy as np
from plotting import show_world, show_path, show_policy, plot_rewards, compare_runs

rng = np.random.default_rng(seed=42)  # makes results repeatable
print("Ready!")

## Part 1: The world

Our **environment** is a 5x5 grid. The **agent** starts in the top-left and wants to reach the goal in the bottom-right.

- `S` = start, `G` = goal (**+10**, episode ends)
- `X` = pit (**-10**, episode ends)
- `#` = wall (you bump into it and stay put)
- Every move costs **-1**, so the agent should want to get there *fast*.

The RL vocabulary:

- **State**: where the agent is (one of the 25 squares)
- **Action**: up, down, left, or right
- **Reward**: the number the environment hands back after each action
- **Episode**: one full attempt, from start until the goal, a pit, or running out of time

In [ ]:
class GridWorld:
    ACTIONS = ["up", "down", "left", "right"]
    MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def __init__(self, layout=None, step_reward=-1, goal_reward=10, pit_reward=-10, max_steps=100):
        self.layout = layout or [
            "S....",
            ".#.X.",
            ".#...",
            ".X.#.",
            "....G",
        ]
        self.rows, self.cols = len(self.layout), len(self.layout[0])
        self.n_states = self.rows * self.cols
        self.n_actions = 4
        self.step_reward, self.goal_reward, self.pit_reward = step_reward, goal_reward, pit_reward
        self.max_steps = max_steps
        self.start = next((r, c) for r in range(self.rows) for c in range(self.cols)
                          if self.layout[r][c] == "S")
        self.reset()

    def to_state(self, pos):
        """Turn a (row, col) position into a single number 0..24."""
        return pos[0] * self.cols + pos[1]

    def reset(self):
        """Start a new episode. Returns the starting state."""
        self.pos = self.start
        self.steps = 0
        return self.to_state(self.pos)

    def step(self, action):
        """Take an action. Returns (next_state, reward, done)."""
        dr, dc = self.MOVES[action]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        # Bumping into the edge or a wall means you stay where you are
        if 0 <= r < self.rows and 0 <= c < self.cols and self.layout[r][c] != "#":
            self.pos = (r, c)
        self.steps += 1

        cell = self.layout[self.pos[0]][self.pos[1]]
        if cell == "G":
            return self.to_state(self.pos), self.goal_reward, True
        if cell == "X":
            return self.to_state(self.pos), self.pit_reward, True
        done = self.steps >= self.max_steps  # out of time
        return self.to_state(self.pos), self.step_reward, done


env = GridWorld()
show_world(env)

### Try moving by hand

Actions are numbers: `0 = up`, `1 = down`, `2 = left`, `3 = right`.
Run the cell below, then change the list of moves and run it again. Can you reach the goal? Can you fall in a pit?

In [ ]:
moves = [3, 3, 3, 1]   # <- edit me! (this one ends badly...)

state = env.reset()
total = 0
for a in moves:
    state, reward, done = env.step(a)
    total += reward
    print(f"{GridWorld.ACTIONS[a]:>5} -> now at {env.pos}, reward {reward:+}")
    if done:
        print("Episode over!")
        break
print("Total reward:", total)

## Part 2: A random agent

Before we make anything smart, let's see how a completely clueless agent does.
It picks a random direction every single step.

In [ ]:
def run_random_episode(env):
    state = env.reset()
    path, total, done = [env.pos], 0, False
    while not done:
        action = rng.integers(env.n_actions)
        state, reward, done = env.step(action)
        path.append(env.pos)
        total += reward
    return path, total

path, total = run_random_episode(env)
print("Total reward:", total)
show_path(env, path, "Random agent")

In [ ]:
# One episode could be luck. Let's run 500 and look at the average.
results, goals = [], 0
for _ in range(500):
    path, total = run_random_episode(env)
    results.append(total)
    goals += env.layout[env.pos[0]][env.pos[1]] == "G"
print(f"Average reward over 500 random episodes: {np.mean(results):.1f}")
print(f"Reached the goal in {goals} of 500 episodes")

Pretty bad. The best possible score here is **+3**: the shortest route is 8 moves, the first 7 cost -1 each, and the last one lands on the goal for +10.
Let's see if learning can get us there.

## Part 3: The agent's brain, the Q-table

The agent keeps a table with one row per state and one column per action:

**Q[state, action] = "how much total future reward do I expect if I take this action here and play well afterward?"**

At first it knows nothing, so everything is 0.

In [ ]:
Q = np.zeros((env.n_states, env.n_actions))
print("Q-table shape:", Q.shape, "  (25 squares x 4 actions)")
print(Q[:5])  # first 5 rows

### Choosing an action: explore vs. exploit

If the agent always picks the action with the highest Q-value (**exploit**), it might never discover a better route.
If it always picks randomly (**explore**), it never uses what it learned.

**Epsilon-greedy** is the simple compromise: with probability `epsilon` pick randomly, otherwise pick the best known action.

In [ ]:
def choose_action(Q, state, epsilon):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))           # explore
    best = np.flatnonzero(Q[state] == Q[state].max())  # all actions tied for best
    return int(rng.choice(best))                       # exploit (break ties randomly)

# Quick check: with an all-zero table, every action is tied, so this is random
print([choose_action(Q, 0, epsilon=0.1) for _ in range(10)])

### Learning: the Q-learning update

After every step, the agent nudges its guess toward a better one:

> **new guess = reward I just got + gamma x (best Q-value from where I landed)**

and then moves the old Q-value a little bit (`alpha`) toward it:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[\, r + \gamma \max_{a'} Q(s',a') - Q(s,a) \Big]$$

- `alpha` (learning rate): how big the nudge is. 0 = never learn, 1 = throw away the old guess.
- `gamma` (discount): how much the agent cares about future rewards vs. right now. 0 = only now, close to 1 = very far-sighted.
- If the episode ended, there is no future, so we just use `r`.

In [ ]:
def q_update(Q, state, action, reward, next_state, done, alpha, gamma):
    future = 0 if done else np.max(Q[next_state])
    target = reward + gamma * future
    Q[state, action] += alpha * (target - Q[state, action])

## Part 4: Train it

The whole algorithm is just: **act, see what happens, update, repeat**, for many episodes.

In [ ]:
def train(env, episodes=500, alpha=0.1, gamma=0.95, epsilon=0.1):
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    for ep in range(episodes):
        state = env.reset()
        total, done = 0, False
        while not done:
            action = choose_action(Q, state, epsilon)
            next_state, reward, done = env.step(action)
            q_update(Q, state, action, reward, next_state, done, alpha, gamma)
            state = next_state
            total += reward
        rewards.append(total)
    return Q, rewards

Q, rewards = train(env, episodes=500)
print(f"First 50 episodes, average reward: {np.mean(rewards[:50]):.1f}")
print(f"Last 50 episodes,  average reward: {np.mean(rewards[-50:]):.1f}")
plot_rewards(rewards)

The grey line jumps around because the agent still explores 10% of the time. The blue line is the trend: it climbs fast and levels off near the best possible score.

## Part 5: What did it learn?

Each arrow shows the action the agent thinks is best in that square. Brighter color = the agent expects more reward from there.

In [ ]:
show_policy(env, Q)

In [ ]:
# Now run the agent with NO exploration (epsilon = 0) to see its best behavior
def run_greedy(env, Q):
    state = env.reset()
    path, total, done = [env.pos], 0, False
    while not done:
        state, reward, done = env.step(int(np.argmax(Q[state])))
        path.append(env.pos)
        total += reward
    return path, total

path, total = run_greedy(env, Q)
print("Total reward:", total)
show_path(env, path, "Trained agent")

### Watch the table fill in

The Q-table starts as all zeros and fills in a little with every move. Let's train a fresh agent and take a snapshot of the table at a few points along the way.

- **Color** = the best value in that square (same scale on every panel).
- **No arrow** = the agent has never updated that square yet, so it has no opinion.

Look for the bright values starting at the goal and spreading back toward the start.

In [ ]:
from plotting import show_progress

def train_with_snapshots(env, checkpoints, alpha=0.1, gamma=0.95, epsilon=0.1):
    Q = np.zeros((env.n_states, env.n_actions))
    snapshots = {}
    for ep in range(1, max(checkpoints) + 1):
        state, done = env.reset(), False
        while not done:
            action = choose_action(Q, state, epsilon)
            next_state, reward, done = env.step(action)
            q_update(Q, state, action, reward, next_state, done, alpha, gamma)
            state = next_state
        if ep in checkpoints:
            snapshots[ep] = Q.copy()   # copy, or every snapshot would be the final table
    return snapshots

snapshots = train_with_snapshots(GridWorld(), checkpoints=[1, 10, 50, 500])
show_progress(GridWorld(), snapshots)

**Questions:** After 1 episode, why do only a few squares have arrows? Why is nothing bright yet? (Hint: one random episode only visits some squares, and it probably never reached the goal.) Between 50 and 500 episodes, which part of the board changes most?

## Part 6: Your turn

Each cell below changes one thing. **Predict what will happen before you run it**, then check.

### Experiment 1: How much should it explore?

In [ ]:
results = {}
for eps in [0.0, 0.1, 0.5]:
    _, r = train(GridWorld(), episodes=500, epsilon=eps)
    results[f"epsilon = {eps}"] = r
compare_runs(results, title="Effect of exploration (epsilon)")

**Questions:** Why does `epsilon = 0.5` level off lower? (Hint: it's still taking a random move half the time, even after it has learned.)
`epsilon = 0` still learns here. Why might never exploring be risky in a bigger, trickier world?

### Experiment 2: How far-sighted should it be?

In [ ]:
results = {}
for g in [0.1, 0.5, 0.95]:
    _, r = train(GridWorld(), episodes=500, gamma=g)
    results[f"gamma = {g}"] = r
compare_runs(results, title="Effect of the discount factor (gamma)")

### Experiment 3: Rewards decide behavior

Let's make the goal worth **0** and a pit only **-2**. Every step still costs -1.
Before running: what do you think the agent will do?

In [ ]:
weird_env = GridWorld(goal_reward=0, pit_reward=-2)
Q_weird, _ = train(weird_env, episodes=500)
path, total = run_greedy(weird_env, Q_weird)
print("Total reward:", total)
show_path(weird_env, path, "Goal = 0, Pit = -2")

It jumps into a pit on purpose! Walking to the goal costs 7 x -1 = **-7**. Diving into the nearest pit costs 3 x -1 - 2 = **-5**.
The agent did exactly what we *rewarded*, not what we *meant*. Designing rewards carefully is one of the hardest parts of real-world RL.

### Experiment 4: Build your own world

Edit the layout below. Use `S`, `G`, `.`, `#` (wall), and `X` (pit). Every row must be the same length.
Make it bigger, add more pits, or build a real maze. You may need more episodes for bigger worlds.

In [ ]:
my_layout = [
    "S..#....",
    ".#.#.##.",
    ".#...#X.",
    ".####.#.",
    "......#G",
]

my_env = GridWorld(layout=my_layout)
show_world(my_env, "My world")
Q_mine, r_mine = train(my_env, episodes=1500)
plot_rewards(r_mine)
show_policy(my_env, Q_mine)

## Recap

- **RL** = an agent learns by trial and error from rewards, with no labeled answers.
- The **Q-table** stores how good each action is in each state.
- The **Q-learning update** nudges each guess toward *reward + discounted best future*.
- **Epsilon-greedy** balances trying new things against using what you know.

**Where this goes next:** a Q-table only works when you can list every state. For things like video games or robots,
we replace the table with a neural network that *estimates* Q-values. That's **Deep Q-Learning (DQN)**, the idea behind
DeepMind's Atari-playing agent. If you want to keep going, check out the
[Gymnasium](https://gymnasium.farama.org/) library and try Q-learning on `FrozenLake-v1`.